# 📘 07-2 심층 신경망 (Deep Neural Network) (1)

In [108]:
from tensorflow import keras

from tensorflow.keras.datasets import fashion_mnist
(train_input, train_target),(test_input,test_target)=fashion_mnist.load_data()

1. 학습 시작
2. loss 확인
3. train vs validation 비교
4. 과적합 체크
5. dropout으로 조정
6. EarlyStopping으로 멈춤
7. best 모델 저장

In [109]:
train_input.shape, train_target.shape

((60000, 28, 28), (60000,))

In [110]:
#정규화
train_scaled=train_input/255.0
train_scaled.shape

(60000, 28, 28)

In [111]:
#Dense 층이 1차원 입력만 받기 때문
train_scaled=train_scaled.reshape(-1,28*28)
train_scaled.shape

(60000, 784)

In [112]:

# test_scaled = test_input / 255.0
# test_scaled = test_scaled.reshape(-1, 28*28)



In [113]:
from sklearn.model_selection import train_test_split
train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)

In [114]:
train_scaled.shape,val_scaled.shape,train_target.shape, val_target.shape

((48000, 784), (12000, 784), (48000,), (12000,))

# 심층 신경망이란?

In [115]:
#시그모이드 활성화 함수를 사용한 은닉층 하나 만들기

dense1=keras.layers.Dense(100, activation='sigmoid',input_shape=(784,))
#sense1은 은닉층잉고 100개의 뉴런을 가진 밀집층이다.
#100: 은닉층은 뉴런의 기준은 경험에 의한 것이며, 출력층의 뉴런보다는 많아야 한다. 

In [116]:
#소프트맥스함수를 사용한 출력층 하나 만들기
dense2=keras.layers.Dense(10,activation='softmax')

# 🛠 5. 심층 신경망 만들기

In [117]:
model=keras.Sequential([dense1,dense2])
# 이 리스트는 은닉층에서 마지막에 출력층의 순서로 나열해야 한다. 
#인공 신경망의 강력한 성능은 이렇게 층을 추가하여
#입력 데이터에 대해 연속적인 학습을 진행하는 능력에서 나온다.

In [118]:
#케라스는 모델의 summanry()메서드를 호출하면 층에 대한 유용한 정보를 얻을 수 있다.
model.summary()

Model: "sequential_17"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_25 (Dense)                │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

In [119]:
# 78,500은 (입력개수*뉴런)+뉴런
# 출력층 1,010dms 100*10+10 임
# total params는 78500+1010임. 입력층과 출력층 더하기 토탈학습해야할 것을 말함

# 🔹 6. 층을 추가하는 다른 방법1 

In [120]:
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Dense(100, activation='sigmoid', input_shape=(784,), name='hidden'),
    keras.layers.Dense(10, activation='softmax', name='output')
], name="패션 MNIST 모델")
# 이 방법은 편리하지만 층을 추가하려면 Sequential클래스 생성자가 매우 길어 진다.
# 또 조건에 따라 층을 추가할 수도 없다.


# 🔹 6. 층을 추가하는 다른 방법2

In [121]:
# Sequentail클래스의 객체를 만들고 이 객체의 add()메서드를 호출하여 층을 추가할 수 있다. 
model=keras.Sequential()
model.add(keras.layers.Dense(100,activation='sigmoid',input_shape=(784,)))
model.add(keras.layers.Dense(10,activation='softmax',name='output'))


In [122]:
model.summary()

Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_27 (Dense)                │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

# 모델을 훈련하자

In [123]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.fit(train_scaled,train_target,epochs=5)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8073 - loss: 0.5661
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8520 - loss: 0.4124
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8639 - loss: 0.3765
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8721 - loss: 0.3529
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8789 - loss: 0.3358


# ⚡ 4. ReLU 함수

In [124]:
model=keras.Sequential()
model.add(keras.layers.Flatten(input_shape=(28,28))) # reshape해도 되고 flatten을 써도 됨. 층이 하나 더 생김
model.add(keras.layers.Dense(100,activation='relu'))
model.add(keras.layers.Dense(10,activation='softmax',name='output'))

c:\Users\06pc-00\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [125]:
model.summary()

Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten_5 (Flatten)             │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

In [126]:
#데이터를 다시 준비
(train_input, train_target),(test_input,test_target)=fashion_mnist.load_data()
train_scaled=train_input/255.0
train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42)

In [127]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['accuracy'])
model.fit(train_scaled,train_target,epochs=5)

Epoch 1/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8114 - loss: 0.5370
Epoch 2/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8573 - loss: 0.3957
Epoch 3/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8719 - loss: 0.3560
Epoch 4/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8787 - loss: 0.3343
Epoch 5/5
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8866 - loss: 0.3181


In [128]:
model.evaluate(val_scaled,val_target)

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8788 - loss: 0.3564


[0.35637128353118896, 0.8788333535194397]

# ⚙️ 7. 옵티마이저 (Optimizer)

In [129]:
# 모델을 학습시키는 방법
#기본경사하강법에는 SGD ->여기에 더 발전해서 모멘텀, -> 더 발전해서 네스트로프모멘텀
#적응적 학습률 옵티마이저에슨 RMSprop, Adam, Adagrad
#적응적 학습률: 모델이 최적점에 가까이 갈수록 학습률을 낮출 수 있다. 

In [130]:
#학습방법 계획 기본경가하강법에는 SGD
model.compile(optimizer='sgd',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
#모델훈련
model.fit(train_scaled,train_target,epochs=5,verbose=0)
model.evaluate(val_scaled, val_target)

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8876 - loss: 0.3185


[0.318535715341568, 0.887583315372467]

In [ ]:
#위와 동일한 코드. 그러나 이건 학습률을 줄수 있음. 
#만약 SGC클래스의 학습률 기본값이 0.01일때 이를 바꾸고 싶다면 learning_rate매개변수에 저장한다.
sgd=keras.optimizers.SGD(learning_rate=0.1)
model.compile(optimizer=sgd,loss='sparse_categorical_crossentropy',metrics=['accuracy'])
#모델훈련
model.fit(train_scaled,train_target,epochs=5,verbose=0)
model.evaluate(val_scaled, val_target)

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8902 - loss: 0.3128


[0.31284642219543457, 0.8901666402816772]

In [134]:
#momentum매개변수의 기본값은 0,0보다 큰값을 지정하면, 그레디언트를 가속도처럼 사용하는 모멘텀최적화를 사용한다. 보당 0.9이상 지정
#nesterov매개변수의 기본값은 False, 이를 True로 바꾸면 nesterov모멘텀최적화(nesterov가속경사)를 사용한다.
sgd=keras.optimizers.SGD(momentum=0.9, nesterov=True)
model.compile(optimizer=sgd,loss='sparse_categorical_crossentropy',metrics=['accuracy'])
#모델훈련
model.fit(train_scaled,train_target,epochs=5,verbose=0)
model.evaluate(val_scaled, val_target)

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8932 - loss: 0.3198


[0.31979700922966003, 0.8931666612625122]

In [136]:
adagrad=keras.optimizers.Adagrad()
model.compile(optimizer=adagrad,loss='sparse_categorical_crossentropy',metrics=['accuracy'])
#모델훈련
model.fit(train_scaled,train_target,epochs=5,verbose=0)
model.evaluate(val_scaled, val_target)

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8948 - loss: 0.3074


[0.30739545822143555, 0.8948333263397217]

In [137]:
rmsprop=keras.optimizers.RMSprop()
model.compile(optimizer=rmsprop,loss='sparse_categorical_crossentropy',metrics=['accuracy'])
#모델훈련
model.fit(train_scaled,train_target,epochs=5,verbose=0)
model.evaluate(val_scaled, val_target)

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8839 - loss: 0.3853


[0.3852582275867462, 0.8839166760444641]

In [ ]:
#가장 많이 사용함. 
adam=keras.optimizers.Adam()
model.compile(optimizer=adam,loss='sparse_categorical_crossentropy',metrics=['accuracy'])
#모델훈련
model.fit(train_scaled,train_target,epochs=5,verbose=0)
model.evaluate(val_scaled, val_target)

375/375 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8880 - loss: 0.3787


[0.37866175174713135, 0.8880000114440918]